# Day 12 — OOP basics & dataclasses
Objectives:
- Create classes with properties and methods.
- Use @dataclass for concise models.
- Inheritance basics.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-12`. Read
`python/ds-60day/companion-guides/day12_oop_dataclasses.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A class defines how a family of objects is constructed and behaves; an
instance is one concrete object. Instance attributes hold per-object
state, and methods are functions retrieved through an instance so that
the instance is passed as `self`. Use a class when state and invariant-
preserving behavior genuinely belong together, not merely to wrap a
dictionary.

A dataclass generates common record behavior such as initialization,
representation, and equality from annotated fields. It does not validate
values automatically. Use `__post_init__` for small invariant checks and
distinguish class attributes shared by all instances from fields owned
by each instance.

### Vocabulary

- **class:** a definition of construction and behavior for related objects.
- **instance:** one concrete object created from a class.
- **attribute:** a named value stored on or resolved through an object.
- **method:** a function accessed through an object, normally receiving `self`.
- **invariant:** a condition that must remain true for a valid object.
- **dataclass:** a class whose record-oriented methods are generated from fields.

## Syntax anatomy

`@dataclass` decorates the following class. In `quantity: int`, the
annotation declares a field and generated constructor parameter.
`self.quantity` refers to the current instance's field. A method that
returns a calculation without mutation is easier to reason about than
one that silently changes several attributes.

### Worked example 1 — Model a validated record with a computed method

Keep line-item state and its subtotal behavior together. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
from dataclasses import dataclass

@dataclass
class LineItem:
    name: str
    unit_price: float
    quantity: int = 1

    def __post_init__(self) -> None:
        if self.unit_price < 0 or self.quantity < 0:
            raise ValueError("price and quantity must be non-negative")

    def subtotal(self) -> float:
        return self.unit_price * self.quantity

item = LineItem("notebook", 4.5, 3)
(item, item.subtotal())

**Expected observation:** `LineItem(name='notebook', unit_price=4.5, quantity=3)` and `13.5`. Construction enforces the invariant.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Show that fields belong to each instance

Mutating one instance should not alter another independent record. Predict first; then run the next cell.

In [ ]:
first = LineItem("pen", 1.5, 2)
second = LineItem("pen", 1.5, 2)
first.quantity = 5
(first.quantity, second.quantity, first == second)

**Expected observation:** `(5, 2, False)`. Each instance owns its quantity; dataclass equality compares current field values.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. When an attribute is missing, inspect whether it was assigned on every constructor path and spell it consistently.
2. Do not use a mutable object as a shared class attribute for per-instance state.
3. Put invariant checks at construction and mutation boundaries, not only in a later calculation.
4. Prefer composition when one object has another; use inheritance only for a genuine substitutable relationship.

**Alternative to compare:** A dictionary suits loose, dynamic records; a named tuple suits immutable records; a dataclass suits named fields with modest behavior and validation.

**Boundary to test:** Negative quantities, mutable default fields, equality after mutation, subclass invariants, and serialization of nested objects need explicit policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class Customer:
    id: int
    name: str
    joined: date

    def age_days(self) -> int:
        return (date.today() - self.joined).days

Customer(1,'Ada', date(2024,1,1)).age_days()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Create a `BankAccount` class with owner and private-by-convention balance state plus `deposit`, `withdraw`, and `balance` behavior. **Contract:** deposits are positive, withdrawals cannot exceed the balance, and invalid operations raise `ValueError` without changing state.
   **Verify:** trace a new account through one deposit, one withdrawal, and two rejected boundary cases.

2. Convert a plain product record to `@dataclass Product(name: str, price: float, quantity: int = 0)`. Add `__post_init__` validation and a `stock_value()` method.
   **Expected behavior:** `Product('tea', 4.0, 3).stock_value() == 12.0`; negative values raise. **Constraint:** use `field(default_factory=...)` if you add any mutable collection field.
   **Verify:** Assert `stock_value()` is `12.0`, equality uses field values, and separate negative-price and negative-quantity constructions raise.

### Additional mastery practice

Put behavior with the data it protects, while keeping ownership and mutation explicit. Prefer composition unless the subtype truly is-a base.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict how a class attribute shared by instances differs from an instance attribute assigned through `self`.
   **Progressive hint:** Class lookup is shared until an instance shadows the name.
   **Verify:** Construct two instances, mutate/shadow one instance field, and assert shared class lookup versus independent instance values explicitly.
4. **Tracing:** Trace dataclass equality for two separately constructed values with equal fields and compare it with object identity using `is`.
   **Progressive hint:** Value equality and identity answer different questions.
   **Verify:** Assert two equal-field dataclass values satisfy `==` but not `is`; mutate or replace a field and confirm equality changes as expected.
5. **Implementation:** Create immutable `OrderLine` and `Order` dataclasses whose total sums quantity × unit price and applies a validated fractional discount.
   **Progressive hint:** Validate non-negative values in `__post_init__`.
   **Verify:** Assert exact order total for multiple lines and a discount, then assert negative quantity/price and out-of-range discount construction each fail.
6. **Debugging:** Repair a dataclass field declared as `items: list[str] = []`.
   **Progressive hint:** Use `field(default_factory=list)` to create one list per instance.
   **Verify:** Create two default instances, mutate one `items` list, and assert the other remains empty and the list objects are not identical.
7. **Edge case and explanation:** Decide whether a discount policy should be a subclass of `Order` or a composed callable; justify the dependency direction.
   **Progressive hint:** A replaceable rule is usually behavior the order uses, not a kind of order.
   **Verify:** Swap two discount callables without changing `Order`; assert totals follow each policy and explain why composition preserves the dependency direction.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Create a `BankAccount` class with owner and private-by-convention balance state plus `deposit`, `withdraw`, and `balance` behavior. **Contract:** deposits are positive, withdrawals cannot exceed the balance, and invalid operations raise `ValueError` without changing state. **Verify:** trace a new account through one deposit, one withdrawal, and two rejected boundary cases.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Create a `BankAccount` class with owner and private-by-convention balance state plus `deposit`, `withdraw`, and `balance` behavior. deposits are positive, withdrawals cannot exc...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Convert a plain product record to `@dataclass Product(name: str, price: float, quantity: int = 0)`. Add `__post_init__` validation and a `stock_value()` method. **Expected behavior:** `Product('tea', 4.0, 3).stock_value() == 12.0`; negative values raise. **Constraint:** use `field(default_factory=...)` if you add any mutable collection field. **Verify:** Assert `stock_value()` is `12.0`, equality uses field values, and separate negative-price and negative-quantity constructions raise.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Convert a plain product record to `@dataclass Product(name: str, price: float, quantity: int = 0)`. Add `__post_init__` validation and a `stock_value()` method. `Product('tea',...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict how a class attribute shared by instances differs from an instance attribute assigned through `self`. **Progressive hint:** Class lookup is shared until an instance shadows the name. **Verify:** Construct two instances, mutate/shadow one instance field, and assert shared class lookup versus independent instance values explicitly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict how a class attribute shared by instances differs from an instance attribute assigned through `self`. Class lookup is shared until an instance shadows the name. Construc...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace dataclass equality for two separately constructed values with equal fields and compare it with object identity using `is`. **Progressive hint:** Value equality and identity answer different questions. **Verify:** Assert two equal-field dataclass values satisfy `==` but not `is`; mutate or replace a field and confirm equality changes as expected.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace dataclass equality for two separately constructed values with equal fields and compare it with object identity using `is`. Value equality and identity answer different que...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Create immutable `OrderLine` and `Order` dataclasses whose total sums quantity × unit price and applies a validated fractional discount. **Progressive hint:** Validate non-negative values in `__post_init__`. **Verify:** Assert exact order total for multiple lines and a discount, then assert negative quantity/price and out-of-range discount construction each fail.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Create immutable `OrderLine` and `Order` dataclasses whose total sums quantity × unit price and applies a validated fractional discount. Validate non-negative values in `__post_...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a dataclass field declared as `items: list[str] = []`. **Progressive hint:** Use `field(default_factory=list)` to create one list per instance. **Verify:** Create two default instances, mutate one `items` list, and assert the other remains empty and the list objects are not identical.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a dataclass field declared as `items: list[str] = []`. Use `field(default_factory=list)` to create one list per instance. Create two default instances, mutate one `items`...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Decide whether a discount policy should be a subclass of `Order` or a composed callable; justify the dependency direction. **Progressive hint:** A replaceable rule is usually behavior the order uses, not a kind of order. **Verify:** Swap two discount callables without changing `Order`; assert totals follow each policy and explain why composition preserves the dependency direction.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Decide whether a discount policy should be a subclass of `Order` or a composed callable; justify the dependency direction. A replaceable rule is usually behavior the order uses,...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
